# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process a dataset using the `mlcroissant` library. All references to record sets, fields, and columns are by their Croissant `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {getattr(metadata, 'name', None)}")
print(f"Dataset description: {getattr(metadata, 'description', None)}")
print(f"Dataset citation: {getattr(metadata, 'citeAs', None)}")


## 2. Data Overview
Review available *record sets*, their fields, and inspect their Croissant `@id`s.

This section lists all record sets, fields, and columns in the dataset schema, referencing their `@id` for unambiguous access.

In [ ]:
# Display all available record sets, their @id, and fields
print("Available record sets (by @id):\n")
for record_set in getattr(metadata, 'recordSet', []):
    rs_id = getattr(record_set, '@id', None)
    name = getattr(record_set, 'name', None)
    print(f"- Record set: {rs_id} (name: {name})")
    fields = getattr(record_set, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        field_id = getattr(field, '@id', None)
        field_name = getattr(field, 'name', None)
        print(f"    - Field: {field_id} (name: {field_name})")
        # List columns if defined:
        cols = getattr(field, 'column', [])
        if cols:
            if not isinstance(cols, list):
                cols = [cols]
            for col in cols:
                col_id = getattr(col, '@id', None)
                col_name = getattr(col, 'name', None)
                print(f"        - Column: {col_id} (name: {col_name})")


## Example records from the first record set
The following cell prints a sample of records using their field `@id` keys. Replace `<record_set_id>` with an actual record set `@id` (see the cell above).

In [ ]:
# List first 2 records from the first available record set
if getattr(metadata, 'recordSet', []):
    first_record_set = metadata.recordSet[0]
    record_set_id = getattr(first_record_set, '@id', None)
    print(f"Showing records for record set @id: {record_set_id}\n")
    for i, x in enumerate(dataset.records(record_set=record_set_id)):
        print(f"Record {i+1}:")
        pprint.pprint(x)
        if i >= 1:
            break
else:
    print("No record sets found in dataset metadata.")

## 3. Data Extraction
Load tabular data from each record set into pandas DataFrames for analysis. All extraction uses record set `@id` and field `@id`.

In [ ]:
# Build list of all record set @ids
record_set_ids = [getattr(rs, '@id', None) for rs in getattr(metadata, 'recordSet', [])]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for record set: {record_set_id}, shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(2))

# For further processing, select the main data table (the largest DataFrame by number of columns/rows)
if dataframes:
    primary_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[1])
    print(f"\nSelected primary record set for EDA: {primary_record_set_id}")
    df = dataframes[primary_record_set_id]
else:
    df = pd.DataFrame()
    primary_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering records, normalizing numeric fields, and grouping. All fields referenced by their Croissant `@id`.

Below, we will:
  - List all numeric fields (`int` or `float`-like columns)
  - Filter by a numeric field, normalize it, and group by a categorical field.

In [ ]:
if not df.empty:
    print(f"DataFrame columns in main record set (@id={primary_record_set_id}):\n{df.columns.tolist()}\n")
    # Try to heuristically select a numeric field
    numeric_candidates = df.select_dtypes(include=['number', 'float', 'int']).columns
    if len(numeric_candidates) == 0:
        # Try to coerce columns to numeric
        converted = {}
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                converted[c] = True
            except (ValueError, TypeError):
                converted[c] = False
        numeric_candidates = [c for c, ok in converted.items() if ok]
    print(f"Numeric field candidates: {numeric_candidates}\n")
    if len(numeric_candidates) > 0:
        numeric_field = numeric_candidates[0]
        print(f"Using '{numeric_field}' as numeric field for demonstration.")

        # Filter records
        threshold = df[numeric_field].mean()  # use mean as example cut-off
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (showing 5):")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}' for filtered records (first 5):")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a grouping field (categorical)
        categorical_cols = df.select_dtypes(include=['object']).columns
        if len(categorical_cols) > 0:
            group_field = categorical_cols[0]
            print(f"\nGrouping filtered data by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame().head()
            print(grouped_df)
        else:
            print("No categorical columns for grouping found.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No tabular record sets to analyze.")

## 5. Visualization

Visualize the distribution of the selected numeric field and the normalized field, if available. All field references by their Croissant `@id` as DataFrame columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and 'numeric_field' in locals():
    plt.figure(figsize=(10, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10, 4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

- Explored the FAIR^2 Croissant dataset for mass spectrometry analysis of human mast cell extracellular vesicles.
- Used `mlcroissant` to load schema, enumerate record sets and fields by `@id`, and extract tabular data.
- Performed EDA by selecting fields by `@id`, filtering and normalizing data, and creating plots.

This workflow can be extended to other record sets and fields by referencing their `@id` as shown.